# Post-build validation

Validates a written time range of the GPM IMERG HH Icechunk store, as called for in
`scaling-design.md` (Verification) and `design.md`:

1. **Completeness scan** — confirm *every* expected timestep in the range was written.
   Uses per-chunk `store.exists(...)` — a metadata HEAD (no data read) that directly
   answers whether the region write committed: a written chunk has an object, an
   unwritten one does not.
2. **Matching spot-check** — sample `N_SAMPLE` random chunks *uniformly across the date
   range* and compare each against the **source HDF5** read natively with `h5py` (an
   independent path from virtualizarr's byte-range derivation). A wrong byte-range
   reference surfaces as a value mismatch; a sampled chunk that was never written is
   surfaced as a gap (so this section also catches missing days, not just bad refs).

> **Run this in `us-west-2`.** Section 1 (metadata only) works anywhere, but the data
> reads in Section 2 — both the store's value reads and the native source reads —
> dereference byte ranges in `gesdisc-cumulus-prod-protected`, which enforces
> same-region access. From a laptop they return `403 AccessDenied`. Earthdata
> credentials must be available (via `.env` or environment).

In [ ]:
import asyncio
import io
import math
import os
import random
import sys
from datetime import datetime, timedelta
from pathlib import Path

import h5py
import icechunk
import numpy as np
import xarray as xr
from obstore.store import S3Store

# Make the processor package importable when running from the repo root.
sys.path.insert(0, str(Path.cwd() / "lambda" / "virtualizarr-processor"))
from virtualizarr_processor import helpers  # noqa: E402
from virtualizarr_processor.processor import _time_index_for  # noqa: E402


def _load_dotenv(path: str = ".env") -> None:
    """Load KEY=VALUE pairs from a local .env into os.environ (does not overwrite)."""
    p = Path(path)
    if not p.exists():
        return
    for line in p.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, val = line.split("=", 1)
        os.environ.setdefault(key.strip(), val.strip().strip('"').strip("'"))


_load_dotenv()
# Populate EARTHDATA_* from env or Secrets Manager.
helpers._load_earthdata_credentials()

# --- Parameters: edit to validate a different range ----------------------------
# Half-open [START, END), matching dispatch.py's _iter_timestamps. The written week
# was dispatched --start 1998-01-01T00:30 --end 1998-01-07T00:00 (last write 23:30 on
# 1998-01-06, idx 287); START=00:00 also lands because of an earlier single-file run.
START = datetime(1998, 1, 1)
END = datetime(1998, 1, 7)  # exclusive
N_SAMPLE = 100
VARIABLES = [
    "precipitation",
    "randomError",
    "precipitationQualityIndex",
    "probabilityLiquidPrecipitation",
]

STEP = timedelta(minutes=30)
EXPECTED_TIMES = [START + k * STEP for k in range(int((END - START) / STEP))]
EXPECTED_IDXS = [_time_index_for(t) for t in EXPECTED_TIMES]
print(
    f"Validating {len(EXPECTED_TIMES)} timesteps: "
    f"{EXPECTED_TIMES[0]:%Y-%m-%d %H:%M} → {EXPECTED_TIMES[-1]:%Y-%m-%d %H:%M} "
    f"(time index {EXPECTED_IDXS[0]}–{EXPECTED_IDXS[-1]}, "
    f"half-open on END={END:%Y-%m-%d %H:%M})"
)

In [ ]:
storage = icechunk.s3_storage(
    bucket="nasa-eodc-public",
    prefix="icechunk/gpmimerg_hh_07",
    region="us-west-2",
    anonymous=True,
)
repo = helpers.open_or_create_repo(storage=storage)
session = repo.readonly_session("main")
store = session.store
ds = xr.open_zarr(
    store, consolidated=False, zarr_version=3, chunks=None, mask_and_scale=False
)

# Open the store read-only (public nasa-eodc-public bucket, anonymous).
# mask_and_scale=False keeps raw -9999.9 sentinels so store values compare
# bit-for-bit against the raw h5py read; chunks=None avoids the dask dependency.
NLON = ds.sizes["lon"]
# lon-chunk size per variable (used by the matching spot-check below).
LON_CHUNK = {v: int(ds[v].encoding["chunks"][1]) for v in VARIABLES}
N_LON_CHUNKS = {v: math.ceil(NLON / cs) for v, cs in LON_CHUNK.items()}
ds

## Section 1 — Completeness scan

For every expected timestep, check that lon-chunk 0 of each variable exists. The region
write for a timestep is atomic, so one chunk's existence proves the whole timestep landed.
Cheap metadata HEADs — runs from anywhere.

In [ ]:
async def _scan_existence(idxs, variables, concurrency=32):
    sem = asyncio.Semaphore(concurrency)

    async def one(var, idx):
        async with sem:
            return (var, idx, await store.exists(f"{var}/c/{idx}/0/0"))

    tasks = [one(var, idx) for var in variables for idx in idxs]
    return await asyncio.gather(*tasks)


results = await _scan_existence(EXPECTED_IDXS, VARIABLES)

idx_to_time = dict(zip(EXPECTED_IDXS, EXPECTED_TIMES))
# (variable, idx) -> written?  Reused by the matching spot-check below.
written = {(var, idx): ok for (var, idx, ok) in results}
missing = [(idx_to_time[idx], var, idx) for (var, idx, ok) in results if not ok]

total = len(results)
print(
    f"Completeness: {total - len(missing)}/{total} chunks present "
    f"across {len(VARIABLES)} variables × {len(EXPECTED_IDXS)} timesteps"
)
if missing:
    print(
        f"\n⚠️  {len(missing)} MISSING "
        "(redrive these timesteps — region writes are idempotent):"
    )
    for t, var, idx in missing[:50]:
        print(f"  {t:%Y-%m-%d %H:%M}  idx={idx:<6d} {var}")
    if len(missing) > 50:
        print(f"  … and {len(missing) - 50} more")
else:
    print("✅ No missing timesteps in range.")

## Section 2 — Matching spot-check (requires `us-west-2` + Earthdata creds)

Sample `N_SAMPLE` random chunks **uniformly across the full date range** (every expected
timestep is equally likely, written or not) and compare each against the source HDF5
granule read natively with `h5py`. The native read uses HDF5's own chunking and
decompression — a path independent of virtualizarr's byte-range derivation — so a wrong
reference surfaces as a value mismatch. Because sampling spans the whole range, a sampled
chunk from a never-written timestep is surfaced as a gap, so this section also flags
missing days proportionally to how much of the range is absent.

In [ ]:
# Source-granule object store (Earthdata-authenticated). Per-timestep byte cache so
# multiple sampled chunks from the same granule reuse a single download.
_src_store = S3Store(
    helpers.BUCKET,
    region="us-west-2",
    credential_provider=helpers._credential_provider(),
)
_granule_cache: dict[int, bytes] = {}


def _granule_bytes(idx: int) -> bytes:
    if idx not in _granule_cache:
        url = helpers.url_for(idx_to_time.get(idx) or (helpers.T0 + idx * STEP))
        key = url.split(f"s3://{helpers.BUCKET}/", 1)[1].replace("//", "/")
        _granule_cache[idx] = _src_store.get(key).bytes().to_bytes()
    return _granule_cache[idx]


def native_chunk(idx: int, var: str, lon_slice: slice) -> np.ndarray:
    """Native h5py read of one variable's lon-block from the source granule."""
    with h5py.File(io.BytesIO(_granule_bytes(idx)), "r") as f:
        # Native granule layout is (time=1, lon=3600, lat=1800).
        return f[f"Grid/{var}"][0, lon_slice, :]

In [ ]:
# Sample N_SAMPLE random (variable, timestep, lon-chunk) triples UNIFORMLY across
# the full range — every expected timestep is equally likely, so missing days get
# sampled in proportion to their share. Group by timestep to reuse downloads.
rng = random.Random(0)
samples = []
for _ in range(N_SAMPLE):
    var = rng.choice(VARIABLES)
    idx = rng.choice(EXPECTED_IDXS)
    c = rng.randrange(N_LON_CHUNKS[var])
    samples.append((var, idx, c))
samples.sort(key=lambda s: s[1])  # cluster by timestep for cache reuse

passed, failures, not_written = 0, [], []
for var, idx, c in samples:
    if not written[(var, idx)]:
        # Never written — flag as a gap; don't compare (store would read fill).
        not_written.append((idx_to_time.get(idx), idx, var, c))
        continue
    cs = LON_CHUNK[var]
    lon_slice = slice(c * cs, min((c + 1) * cs, NLON))
    store_block = ds[var].isel(time=idx, lon=lon_slice).values
    native_block = native_chunk(idx, var, lon_slice)
    if np.array_equal(store_block, native_block):
        passed += 1
    else:
        max_diff = float(
            np.nanmax(np.abs(store_block.astype("f8") - native_block.astype("f8")))
        )
        failures.append((idx_to_time.get(idx), idx, var, c, max_diff))

compared = passed + len(failures)
print(
    f"Matching spot-check: sampled {len(samples)} chunks uniformly across the range "
    f"(downloaded {len(_granule_cache)} granules)"
)
print(f"  compared (written): {passed}/{compared} match source bytes")
print(f"  sampled but NOT written: {len(not_written)}")
if not_written:
    print(
        "\n⚠️  Sampled chunks that were never written (range has gaps — see Section 1):"
    )
    print(f"  {'timestamp':<20}{'idx':>8}  {'variable':<32}{'lon-chunk':>10}")
    for t, idx, var, c in not_written[:50]:
        ts = f"{t:%Y-%m-%d %H:%M}" if t else "?"
        print(f"  {ts:<20}{idx:>8}  {var:<32}{c:>10}")
    if len(not_written) > 50:
        print(f"  … and {len(not_written) - 50} more")
if failures:
    print(
        "\n❌ VALUE MISMATCHES (bad byte-range reference — investigate these granules):"
    )
    print(
        f"  {'timestamp':<20}{'idx':>8}  "
        f"{'variable':<32}{'lon-chunk':>10}{'max|diff|':>14}"
    )
    for t, idx, var, c, d in failures:
        ts = f"{t:%Y-%m-%d %H:%M}" if t else "?"
        print(f"  {ts:<20}{idx:>8}  {var:<32}{c:>10}{d:>14.4g}")
if not failures and not not_written:
    print("\n✅ All sampled chunks were written and match the source HDF5.")

## Section 3 — Redrive missing timesteps

Turn Section 1's missing list into targeted SQS sends: one message per missing
timestep (deduped across variables), in the body shape the processor handler
expects (same as `dispatch.py`). Region writes are idempotent, so this is safe and
covers every gap whether or not it reached the DLQ. Leave `REDRIVE=False` for a dry
run; set `QUEUE_URL` and `REDRIVE=True` to actually enqueue.

In [ ]:
import json
import re

import boto3

QUEUE_URL = ""  # e.g. https://sqs.us-west-2.amazonaws.com/<acct>/gpmimerg-vz-dp-queue
REGION = "us-west-2"
REDRIVE = False  # False = dry run (summary + sample body); True = actually send

_MULTISLASH = re.compile(r"/{2,}")


def _message_body(t: datetime) -> str:
    """SQS body in the shape the handler expects; collapse url_for's doubled '/'."""
    url = helpers.url_for(t)
    prefix = f"s3://{helpers.BUCKET}/"
    key = _MULTISLASH.sub("/", url[len(prefix) :])
    return json.dumps(
        {
            "Records": [
                {"s3": {"bucket": {"name": helpers.BUCKET}, "object": {"key": key}}}
            ]
        }
    )


# Unique missing time indices (dedupe the per-variable entries from Section 1).
missing_idxs = sorted({idx for (_t, _v, idx) in missing})
missing_times = [idx_to_time[idx] for idx in missing_idxs]
print(f"{len(missing_idxs)} missing timesteps to redrive")
if missing_times:
    print(
        f"  range {missing_times[0]:%Y-%m-%d %H:%M} … "
        f"{missing_times[-1]:%Y-%m-%d %H:%M}"
    )

if not missing_times:
    print("Nothing missing — nothing to redrive.")
elif not REDRIVE:
    print("\nREDRIVE=False — dry run, nothing sent.")
    print(f"sample body: {_message_body(missing_times[0])}")
elif not QUEUE_URL:
    raise ValueError("Set QUEUE_URL (and REDRIVE=True) to send.")
else:
    sqs = boto3.client("sqs", region_name=REGION)
    sent = 0
    for i in range(0, len(missing_times), 10):  # SQS send_message_batch max = 10
        chunk = missing_times[i : i + 10]
        entries = [
            {"Id": str(j), "MessageBody": _message_body(t)} for j, t in enumerate(chunk)
        ]
        for _ in range(4):  # retry partial failures
            failed = sqs.send_message_batch(QueueUrl=QUEUE_URL, Entries=entries).get(
                "Failed", []
            )
            if not failed:
                break
            failed_ids = {f["Id"] for f in failed}
            entries = [e for e in entries if e["Id"] in failed_ids]
        else:
            raise RuntimeError(f"batch still failing after retries: {failed[:3]}")
        sent += len(chunk)
    print(f"\nRedriven {sent} messages to the queue.")

## Interpreting results

- **0 missing (Section 1) + all sampled chunks written and matched (Section 2)** ⇒ the
  range is fully written and the chunk references resolve to the correct source bytes.
- **Missing entries (Section 1) / sampled-but-not-written (Section 2)** ⇒ redrive those
  timesteps; region writes are idempotent so re-dispatching is safe. Section 1 is the
  authoritative, exhaustive list; Section 2 surfaces the same gaps probabilistically as
  a cross-check on the sampling.
- **Value mismatches (Section 2)** ⇒ a chunk's virtual byte-range reference is wrong;
  investigate the listed granule(s).

In [ ]:
#